# Lab 2 · Your first model

**What you'll build:** a loss function, an R², and a line fitted by brute force —
all by hand, before you're allowed to call scikit-learn.

The goal is not the line. The goal is that when you later type
`RandomForestRegressor()`, you know what it is doing to your data.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labgrader import panel, grade_lab, TARGET, CLIMATE

df = panel()
print(df.shape, "rows x columns")
print("target column:", TARGET)

## 1. A model is a function with knobs

That is the entire idea. Here is the simplest one that could possibly work:
predict a corridor's CDEI from its summer precipitation.

$$\hat{y} = w \cdot \text{PPT} + b$$

`w` and `b` are the knobs. Training a model means *choosing the knobs*.

In [ ]:
x = df["PPT_sm"].to_numpy()     # summer precipitation, mm
y = df[TARGET].to_numpy()       # CDEI

def predict(x, w, b):
    return w * x + b

guess = predict(x, w=0.0001, b=0.0)
print("first 5 predictions:", guess[:5].round(4))
print("first 5 truths     :", y[:5].round(4))

## 2. How wrong is it? — the loss

You need one number that says how bad a set of knobs is. **Mean squared error**
is the usual choice: average the squared gaps.

$$\text{MSE} = \frac{1}{n}\sum_i (y_i - \hat{y}_i)^2$$

Squared, not absolute, for two reasons: it punishes a few big misses more than
many small ones, and it is smooth, which makes it easy to minimise.

### ✏️ Assignment 1 — `mse`

In [ ]:
def mse(y_true, y_pred):
    """Mean squared error."""
    # >>> YOUR TURN
    raise NotImplementedError


# mse(y, predict(x, 0.0001, 0.0))  -> some small number

## 3. Is that *good*? — R²

MSE alone is unreadable: 0.0002 is meaningless without a scale. So compare
against the dumbest model that exists — **always predict the mean**.

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- $R^2 = 1$ — perfect.
- $R^2 = 0$ — exactly as good as predicting the mean. Worthless, but not harmful.
- $R^2 < 0$ — **worse than predicting the mean.** This is possible, it happens in
  this project, and it is the single most important number in the whole result.

Most tutorials never show you a negative R². Yours will be negative.

### ✏️ Assignment 2 — `r2`

In [ ]:
def r2(y_true, y_pred):
    """Coefficient of determination. Negative is allowed and meaningful."""
    # >>> YOUR TURN
    raise NotImplementedError


# Sanity: predicting the mean must give exactly 0.0
# r2(y, np.full_like(y, y.mean()))

## 4. Turning the knobs

You have a loss. Now find the knobs that minimise it. Two ways:

**Brute force.** Try lots of `w`, and for each one take the `b` that is
automatically best. There is a shortcut: for any slope, the best intercept is
always `mean(y) - w * mean(x)`, because the least-squares line always passes
through the centre of the data. That turns a 2-D search into a 1-D one.

**Closed form.** For a straight line you can just *solve* for the answer —
that is `np.polyfit(x, y, 1)`. Lab 4 does this properly with matrices.

Do it by search first. Seeing the loss curve bottom out is worth more than the
formula.

### ✏️ Assignment 3 — `best_line`

Return `(w, b)` minimising MSE. Search `w` over a grid, use the intercept
shortcut, keep the best.

Hint for the grid: precipitation is in the hundreds of mm and CDEI is around
0.01, so the slope is *tiny*. `np.linspace(-1e-3, 1e-3, 4001)` is a sane range.
If your answer lands on the edge of the grid, widen it.

In [ ]:
def best_line(x, y):
    """Slope and intercept minimising mean squared error."""
    # >>> YOUR TURN
    raise NotImplementedError

In [ ]:
w, b = best_line(x, y)
print(f"your fit    : w = {w:.3e}   b = {b:.5f}   MSE = {mse(y, predict(x, w, b)):.6e}")

pw, pb = np.polyfit(x, y, 1)
print(f"np.polyfit  : w = {pw:.3e}   b = {pb:.5f}   MSE = {mse(y, predict(x, pw, pb)):.6e}")
print(f"\nR2 of your line: {r2(y, predict(x, w, b)):+.4f}")

In [ ]:
# The loss curve. This is what "training" is walking down.
ws = np.linspace(-1e-3, 1e-3, 400)
losses = [mse(y, predict(x, wi, y.mean() - wi * x.mean())) for wi in ws]

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(ws, losses, lw=2)
ax.axvline(w, color="crimson", ls="--", label=f"your best w = {w:.2e}")
ax.set_xlabel("slope w"); ax.set_ylabel("MSE"); ax.legend()
ax.set_title("The loss surface, sliced along the slope")
plt.show()

## 5. Now the one-liner

Everything above is what scikit-learn does when you call `.fit()`.

In [ ]:
from sklearn.linear_model import LinearRegression

m = LinearRegression().fit(x.reshape(-1, 1), y)
print(f"sklearn     : w = {m.coef_[0]:.3e}   b = {m.intercept_:.5f}")
print(f"sklearn R2  : {m.score(x.reshape(-1, 1), y):+.4f}")

> **Look at that R².** It is barely above zero, and this is the *training* score —
> the model was scored on the very data it was fitted to, which is the easiest
> possible test. It has not been asked to predict anything new yet.
>
> Lab 3 asks it to.

---
## Grade it

In [ ]:
grade_lab(2, globals())

### What you should be able to say out loud

- Training = choosing knobs to minimise a loss.
- MSE is the loss; R² is MSE made readable by comparing to "predict the mean".
- **Negative R² means worse than the mean**, and it is a real result, not a bug.
- A score on the training data is not evidence of anything.